
# Posterior predictive check: Bayesian goodness-of-fit diagnostic

The posterior predictive check (PPC) is the gold-standard Bayesian
goodness-of-fit diagnostic (Rubin 1984; Gelman et al. 1996). We fit mock
SDSS photometry, draw 100 samples from the posterior, regenerate mock
photometry for each sample, and overlay the band-by-band model envelope
(16th, 50th, 84th percentiles) against observed data with residuals
normalized by noise. For well-fit models, residuals cluster within ±2σ.

This demonstrates tengri's differentiable + JIT path that makes 100
posterior re-predictions negligible in cost.

Reference: Rubin 1984, J. Educ. Stat., 9, 26; Gelman et al. 1996,
Bayesian Data Analysis (Chapman & Hall).


In [ ]:
import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# Load SSP data and define observation
ssp = tengri.load_ssp()
obs = tengri.Observation(
    photometry=tengri.Photometry.from_names(["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z"])
)

# Build model with 7 free parameters
model = tengri.SEDModel.build(
    ssp,
    observation=obs,
    sfh={"type": "tsnorm", "*": tengri.FREE},
    dust={
        "type": "two_component",
        "*": tengri.FIXED,
        "tau_diff": tengri.Uniform(0.0, 1.5),
        "slope": -0.7,
    },
    redshift=tengri.Fixed(0.1),
)

# --- Generate mock photometry (star-forming galaxy) ---
key = jax.random.PRNGKey(42)
truth = dict(model.spec.sample(key))
truth.update(
    sfh_tsnorm_peak_lbt_gyr=3.0,
    sfh_tsnorm_width_gyr=2.0,
    sfh_tsnorm_log_peak_sfr=1.0,
    sfh_tsnorm_skew=0.3,
    sfh_tsnorm_trunc=10.0,
    met_logzsol=-0.2,
    dust_tau_diff=0.5,
)
mock = model.mock(truth, snr=25.0, key=key)

# --- Fit with variational inference (geoVI: nonlinear posterior) ---
# geoVI provides samples from a nonlinear approximation to the true posterior,
# capturing parameter degeneracies. The model envelope will be accurate.
forward = tengri.ForwardModel.build(sed=model, observation=obs)
posterior = forward.fit(
    mock.flux_obs,
    mock.noise,
    method="native_vi_nonlinear",
    n_iterations=500,
    n_samples=3,
    n_posterior_samples=100,  # Posterior samples for PPC
    verbose=False,
)


# --- Generate posterior predictive samples ---
# For each posterior sample, regenerate mock photometry to build the envelope.
# Using vmap over the posterior samples for JIT efficiency.
def predict_one_sample(param_dict):
    """Predict photometry for a single posterior sample."""
    return jnp.asarray(model.predict_photometry(param_dict))


# Convert posterior samples dict to a list of parameter dicts
sample_keys = list(posterior.samples.keys())
n_samples = posterior.samples[sample_keys[0]].shape[0]

param_dicts = []
for i in range(n_samples):
    param_i = {k: posterior.samples[k][i] for k in sample_keys}
    param_dicts.append(param_i)

# Predict photometry for all posterior samples (vectorized)
pred_envelope = np.array([predict_one_sample(p) for p in param_dicts])  # (n_samples, n_bands)

# Compute percentiles: 16th, 50th, 84th
p16 = np.percentile(pred_envelope, 16, axis=0)
p50 = np.percentile(pred_envelope, 50, axis=0)
p84 = np.percentile(pred_envelope, 84, axis=0)

# --- Plot: PPC with residuals ---
wave_eff = np.array([3551, 4686, 6166, 7480, 8932])  # SDSS effective wavelengths [Angstrom]
band_names = ["u", "g", "r", "i", "z"]
flux_obs = np.array(mock.flux_obs)
noise = np.array(mock.noise)

fig, (ax_phot, ax_res) = plt.subplots(
    2, 1, figsize=(8, 6), height_ratios=[3, 1], sharex=True, gridspec_kw={"hspace": 0.05}
)

# Panel 1: Observed photometry vs model envelope
ax_phot.fill_between(
    wave_eff, p16, p84, alpha=0.3, color="C0", label="Posterior 68% envelope (geoVI)"
)
ax_phot.plot(wave_eff, p50, color="C0", lw=2.0, label="Posterior median")
ax_phot.errorbar(
    wave_eff,
    flux_obs,
    yerr=noise,
    fmt="o",
    color="k",
    ms=7,
    capsize=4,
    elinewidth=1.5,
    label="Observed",
    zorder=5,
)
ax_phot.scatter(
    wave_eff,
    np.array(mock.flux_true),
    marker="s",
    s=60,
    facecolors="none",
    edgecolors="C3",
    lw=1.5,
    label="Truth",
    zorder=4,
)
ax_phot.set_ylabel(r"$f_\nu$ [arbitrary]")
ax_phot.legend(fontsize=10, frameon=False, loc="upper right")
ax_phot.set_title("Posterior Predictive Check: Mock Photometry")
ax_phot.grid(True, alpha=0.2)

# Panel 2: Residuals in units of noise
# Good PPC: residuals within ±2σ for well-fit model
residuals_median = (flux_obs - p50) / noise
residuals_lo = (flux_obs - p84) / noise
residuals_hi = (flux_obs - p16) / noise

ax_res.axhline(0, color="0.5", ls="--", lw=1.0, alpha=0.7)
ax_res.axhline(2, color="0.5", ls=":", lw=0.8, alpha=0.5)
ax_res.axhline(-2, color="0.5", ls=":", lw=0.8, alpha=0.5)
ax_res.fill_between(wave_eff, residuals_lo, residuals_hi, alpha=0.3, color="C0")
ax_res.scatter(wave_eff, residuals_median, color="C0", s=50, zorder=5)
ax_res.set_xlabel(r"Wavelength [$\AA$]")
ax_res.set_ylabel(r"$(f_\mathrm{obs} - f_\mathrm{med}) / \sigma$")
ax_res.set_ylim(-3.5, 3.5)
ax_res.set_xticks(wave_eff)
ax_res.set_xticklabels(band_names)
ax_res.grid(True, alpha=0.2, axis="y")

fig.tight_layout()
plt.savefig("plot_posterior_predictive_check.png", dpi=150, bbox_inches="tight")

# --- Summary statistics ---
max_residual = np.max(np.abs(residuals_median))
within_2sigma = np.sum(np.abs(residuals_median) < 2.0) / len(residuals_median)
print("Posterior predictive check diagnostics:")
print(f"  Max |residual| (σ units): {max_residual:.2f}")
print(f"  Fraction within ±2σ: {within_2sigma:.1%}")
print(f"  → Well-fit model: {within_2sigma > 0.95}")